In [ ]:
"""
Objetivo: Utilizar o método Autoregressive Distributed Lag (ARDL) e o Bounds Test de Pesaran 
para estimar o Mecanismo de Correção de Erros (ECM) e mapear os determinantes macroeconômicos
de curto e longo prazo para cada segmento de crédito.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.ardl import ardl_select_order, UECM
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. CARREGAMENTO E CORTE DA AMOSTRA
# ==========================================
def carregar_dados_ardl(arquivo_csv: str) -> pd.DataFrame:
    """Carrega os dados e aplica o corte a partir de 2005 para estabilidade dos testes em nível."""
    print("[*] Carregando a base de dados limpa...")
    df = pd.read_csv(arquivo_csv, index_col=0, parse_dates=True)
    df_model = df.loc['2005-01-01':'2026-04-01'].copy()
    return df_model

# ==========================================
# 2. ESPECIFICAÇÃO E ESTIMAÇÃO DOS MODELOS ECM
# ==========================================
def estimar_ardl_ecm(df: pd.DataFrame, especificacoes: dict):
    """
    Realiza seleção automática de lags (BIC), estima o modelo de correção de erros (UECM) 
    e extrai os coeficientes de longo prazo (Bounds Test) e curto prazo.
    """
    print("\n[*] Iniciando a rotina de seleção automática ARDL e estimação ECM...\n")
    
    for nome_carteira, specs in especificacoes.items():
        print(f"=======================================================")
        print(f" CARTEIRA: {nome_carteira.upper()}")
        print(f"=======================================================")
        
        y = df[specs['y']].dropna()
        X = df[specs['x']].dropna()
        
        # Sincronização do index
        comum = y.index.intersection(X.index)
        y, X = y.loc[comum], X.loc[comum]
        
        # Seleção estrita de ordem de defasagem (BIC)
        selecao = ardl_select_order(endog=y, maxlag=4, exog=X, maxorder=2, ic='bic', trend='c')
        
        if not selecao.dl_lags:
            print("[!] Nenhuma variável macro sobreviveu ao filtro do BIC.")
            continue
        
        ordem_x_dict = {col: max(1, max(lags)) for col, lags in selecao.dl_lags.items()}
        X_selecionado = X[list(selecao.dl_lags.keys())]
        ordem_y = max(1, max(selecao.ar_lags)) if selecao.ar_lags is not None else 1
        
        print(f"[*] Melhor arquitetura (BIC): ARDL({ordem_y}, {ordem_x_dict})")
        
        # Instanciação do Unrestricted Error Correction Model (UECM)
        modelo_ecm = UECM(endog=y, lags=ordem_y, exog=X_selecionado, order=ordem_x_dict, trend='c')
        fit_ecm = modelo_ecm.fit()
        
        # Bounds Test de Pesaran
        teste_bounds = fit_ecm.bounds_test(case=3)
        
        # Captura do ECT (Error Correction Term)
        chave_ect = specs['y'] + '.L1'
        if chave_ect not in fit_ecm.params:
            chave_ect = [c for c in fit_ecm.params.index if '.L1' in c and specs['y'] in c][0]
            
        param_ect = fit_ecm.params[chave_ect]
        p_valor_ect = fit_ecm.pvalues[chave_ect]
        
        print(f"[*] Parâmetro de Correção (Lambda/ECT): {param_ect:.4f} (P-valor: {p_valor_ect:.4f})")
        print(f"[*] Estatística F do Bounds Test: {teste_bounds.stat:.2f}")
        
        if param_ect < 0 and p_valor_ect < 0.05:
            meia_vida_ecm = np.log(0.5) / param_ect
            print(f"[*] Status: Cointegração validada. A carteira absorve choques e volta ao equilíbrio pela metade em {meia_vida_ecm:.2f} trimestres.")
        else:
            print("[!] Alerta: ECT instável ou sem cointegração estatística robusta a 5%.")
            
        print("\n--- Principais Determinantes de Curto Prazo (Variações) ---")
        curto_prazo = fit_ecm.params.filter(like='D.')
        p_valores_cp = fit_ecm.pvalues.filter(like='D.')
        
        for var, coef in curto_prazo.items():
            if p_valores_cp[var] < 0.10: 
                print(f" > {var}: {coef:.4f} (Sig: {p_valores_cp[var]:.3f})")
        print("\n")
        
if __name__ == "__main__":
    df_model = carregar_dados_ardl('base_macro_fred_limpa.csv')
    
    modelos_especificacao = {
        'Hipotecas': {
            'y': 'inad_hipotecas',
            'x': ['desemprego', 'fed_funds', 'spread_10y3m', 'preco_imoveis', 'esforco_pagamento', 'sloos_aperto']
        },
        'Consumidor': {
            'y': 'inad_consumidor',
            'x': ['desemprego', 'fed_funds', 'spread_10y3m', 'venda_veiculos', 'esforco_pagamento', 'sloos_aperto']
        },
        'Cartões': {
            'y': 'inad_cartoes',
            'x': ['desemprego', 'fed_funds', 'spread_10y3m', 'esforco_pagamento', 'sloos_aperto']
        }
    }
    
    estimar_ardl_ecm(df_model, modelos_especificacao)
    print("[SUCCESS] Pipeline ARDL-ECM concluído.")